# IMPORT LIBRARY

In [113]:
import pandas as pd
import numpy as np
import random
from collections import Counter

# PARAMETER 

In [114]:
POPULASI = 50
GENERASI = 100

PROBABILITAS_CROSSOVER = 0.7
PROBABILITAS_MUTASI = 0.4
BIAS_CROSSOVER = 0.85
BIAS_MUTASI = 0.85

VIOLATION_COST = 100
TOURNAMENT_SIZE = 10

# DATASET

In [115]:
guru_df = pd.read_csv('../dataset/guru.csv')
kelas_df = pd.read_csv('../dataset/kelas.csv')
mapel_df = pd.read_csv('../dataset/mapel.csv')
relasi_guru_mapel_df = pd.read_csv('../dataset/relasi_guru_mapel.csv')
slot_df = pd.read_csv('../dataset/slot.csv')

# RELASI

In [116]:
join = (
    relasi_guru_mapel_df
    .merge(guru_df, on="guru_id", how="left")
    .merge(mapel_df, on="mapel_id", how="left")
)

In [117]:
join["total"] = join.groupby("guru_id")["durasi"].transform("sum")

In [118]:
mapping_hari = {
    "Senin" : 1,
    "Selasa" : 2,
    "Rabu" : 3,
    "Kamis" : 4,
    "Jumat" : 5
}

In [119]:
relasi = join[[
    "guru_id", "mapel_id", "jam_per_minggu", "tingkatan", 
    "durasi", "total", "MGMP"
]].copy()

In [120]:
relasi["MGMP"] = relasi["MGMP"]. map(mapping_hari)

In [121]:
mapel_jam = dict(zip(mapel_df["mapel_id"], mapel_df["jam_per_minggu"]))

In [122]:
slot_per_hari = {
    1:8,
    2:8,
    3:8,
    4:7,
    5:5
}

# DICTIONARY

In [123]:
total_kelas = kelas_df["kelas_id"].count()

In [124]:
batas_siang = {0: 5, 1: 5, 2: 4, 3: 5, 4: 4}
batas_mgmp = {0: 2, 1: 2, 2: 2, 3: 2, 4: 1}

In [125]:
guru_by_mapel = (
    relasi
    .groupby(["mapel_id", "tingkatan"])["guru_id"]
    .apply(list)
    .to_dict()
)

In [126]:
MAPEL_LIST = list(range(1, 14))

# INISIALISASI INDIVIDU

In [127]:
def individu_construct(slot_per_hari, total_kelas, guru_by_mapel, relasi):

    mapel_id = list(range(1, 14))     # 13 mapel
    tingkatan = relasi["tingkatan"].iloc[0]

    # preprocess guru per mapel
    guru_mapel_list = {
        m: guru_by_mapel.get((m, tingkatan), [])
        for m in mapel_id
    }

    individu = []

    for _ in range(total_kelas):

        kelas = []

        # ===== GEN MAPEL PER HARI =====
        for hari in sorted(slot_per_hari.keys()):  # 1..5
            jumlah_slot = slot_per_hari[hari]
            gen_hari = [random.choice(mapel_id) for _ in range(jumlah_slot)]
            kelas.append(gen_hari)

        # ===== GEN GURU (13 MAPEL) =====
        gen_guru = [
            random.choice(guru_mapel_list[m]) if guru_mapel_list[m] else 0
            for m in mapel_id
        ]

        kelas.append(gen_guru)

        individu.append(kelas)

    return individu


In [128]:
populasi = []
for i in range(POPULASI):
    individu = individu_construct(slot_per_hari, total_kelas, guru_by_mapel, relasi)
    populasi.append(individu)


# EVALUASI

In [129]:
def guru_bentrok(individu, VIOLATION_COST):
    pelanggaran = 0
    cost = 0

    slot_harian = individu[0][:-1]

    for hari_id, hari in enumerate(slot_harian):
        for jam_id in range(len(hari)):
            guru_used = set()

            for kelas in individu:
                mapel = kelas[hari_id][jam_id]

                if mapel == 0:
                    continue

                guru = kelas[-1][mapel - 1]

                if guru in guru_used:
                    pelanggaran += 1
                    cost += VIOLATION_COST
                else:
                    guru_used.add(guru)

    return pelanggaran, cost

In [130]:
def konfigurasi_mapel(individu, mapel_jam, VIOLATION_COST):
    pelanggaran = 0
    cost = 0

    for kelas in individu:
        slot_harian = kelas[:-1]  # senin–jumat

        mapel_hari = {}

        for hari_id, hari in enumerate(slot_harian):
            for slot_id, mapel in enumerate(hari):
                if mapel not in mapel_hari:
                    mapel_hari[mapel] = {}
                if hari_id not in mapel_hari[mapel]:
                    mapel_hari[mapel][hari_id] = []
                mapel_hari[mapel][hari_id].append(slot_id)

        # evaluasi per mapel
        for mapel_id, distribusi_hari in mapel_hari.items():
            total_jam = mapel_jam.get(mapel_id, 0)

            # rule jumlah hari
            hari_terpakai = [len(v) for v in distribusi_hari.values()]

            if total_jam in (2, 3):
                # mapel dengan 2 , 3 jam per minggu
                if len(distribusi_hari) != 1:
                    pelanggaran += 1
                    cost += VIOLATION_COST

            elif total_jam == 4:
                # mapel dengan 4 jam per minggu
                if sorted(hari_terpakai) != [2, 2]:
                    pelanggaran += 1
                    cost += VIOLATION_COST

            elif total_jam == 5:
                # mapel dengan 5 jam per minggu
                if sorted(hari_terpakai) != [2, 3]:
                    pelanggaran += 1
                    cost += VIOLATION_COST

            # mapel harus pada slot yang berdekatan pada hari yang sama
            for slot_list in distribusi_hari.values():
                if len(slot_list) > 1:
                    slot_list = sorted(slot_list)
                    for i in range(len(slot_list) - 1):
                        if slot_list[i + 1] - slot_list[i] != 1:
                            pelanggaran += 1
                            cost += VIOLATION_COST
                            break

    return pelanggaran, cost

In [131]:
def mapel_pjok(individu, batas_siang, VIOLATION_COST):
    pelanggaran = 0
    cost = 0

    for kelas in individu:
        slot_harian = kelas[:-1] # hari saja

        for hari_id, hari in enumerate(slot_harian):
            batas = batas_siang[hari_id]

            for slot_id, mapel in enumerate(hari):
                if mapel == 8 and slot_id > batas:
                    pelanggaran += 1
                    cost += VIOLATION_COST
                    
    return pelanggaran, cost

In [132]:
def durasi_guru(individu, relasi, VIOLATION_COST):
    pelanggaran = 0
    cost = 0

    # lookup (guru, mapel) -> durasi
    durasi_lookup = {
        (row.guru_id, row.mapel_id) : row.durasi
        for row in relasi.itertuples(index=False)
    }

    for kelas in individu:
        slot_harian = kelas[:-1]
        guru_mapel = kelas[-1]

    # menghitung total jam
    jam_guru_aktual = Counter()

    for hari in slot_harian:
        for mapel in hari:
            guru = guru_mapel[mapel - 1]
            jam_guru_aktual[(guru, mapel)] += 1

    for (guru, mapel), jam_aktual in jam_guru_aktual.items():
        durasi_wajib = durasi_lookup.get((guru, mapel))

        if durasi_wajib is None:
            # tidak valid di relasi.df
            pelanggaran += 1
            cost += VIOLATION_COST

        elif jam_aktual != durasi_wajib:
            pelanggaran += abs(jam_aktual - durasi_wajib)
            cost += VIOLATION_COST * abs(jam_aktual - durasi_wajib)

    return pelanggaran, cost

In [133]:
def mapel_jam_per_minggu(individu, mapel_jam, VIOLATION_COST):
    pelanggaran = 0
    cost = 0

    for kelas in individu:
        slot_harian = kelas[:-1]

        semua_slot = []
        for hari in slot_harian:
            semua_slot.extend(hari)

        hitung_mapel = Counter(semua_slot)

        for mapel_id, jam_wajib in mapel_jam.items():
            jam_aktual = hitung_mapel.get(mapel_id, 0)

            if jam_aktual != jam_wajib:
                diff = abs(jam_aktual - jam_wajib)

                pelanggaran += diff
                cost += diff * VIOLATION_COST
                
    return pelanggaran, cost

In [134]:
def build_mapel_mgmp(relasi):
    return {
        row.mapel_id: row.MGMP - 1
        for row in relasi.itertuples(index=False)
        if row.MGMP is not None
    }
    
def mgmp_slot(individu, relasi, batas_mgmp, VIOLATION_COST):
    pelanggaran = 0
    cost = 0

    mapel_mgmp = build_mapel_mgmp(relasi)

    for kelas in individu:
        slot_harian = kelas[:-1]

        for hari_id, hari in enumerate(slot_harian):
            for slot_id, mapel in enumerate(hari):

                mgmp_hari = mapel_mgmp.get(mapel)

                if mgmp_hari is None:
                    continue

                if hari_id == mgmp_hari:
                    batas_slot = batas_mgmp[hari_id]

                    if slot_id > batas_slot:
                        pelanggaran += 1
                        cost += VIOLATION_COST
                        
    return pelanggaran, cost


In [135]:
EVALUATORS = [
    (guru_bentrok,        {}),
    (konfigurasi_mapel,  {"mapel_jam": mapel_jam}),
    (mapel_pjok,         {"batas_siang": batas_siang}),
    (durasi_guru,        {"relasi": relasi}),
    (mapel_jam_per_minggu, {"mapel_jam": mapel_jam}),
    (mgmp_slot,          {"relasi": relasi, "batas_mgmp": batas_mgmp}),
]

In [136]:
def evaluator_function(individu):
    pelanggaran_total = 0
    cost_total = 0

    for func, params in EVALUATORS:
        pelanggaran, cost = func(individu, **params, VIOLATION_COST=VIOLATION_COST)
        pelanggaran_total += pelanggaran
        cost_total += cost
    
    return pelanggaran_total, cost_total


In [137]:
for individu in populasi:
    pelanggaran, cost = evaluator_function(individu)
    print("Pelanggaran: ", pelanggaran)
    print("Cost: ", cost)

Pelanggaran:  1623
Cost:  162300
Pelanggaran:  1504
Cost:  150400
Pelanggaran:  1653
Cost:  165300
Pelanggaran:  1576
Cost:  157600
Pelanggaran:  1529
Cost:  152900
Pelanggaran:  1525
Cost:  152500
Pelanggaran:  1617
Cost:  161700
Pelanggaran:  1560
Cost:  156000
Pelanggaran:  1555
Cost:  155500
Pelanggaran:  1613
Cost:  161300
Pelanggaran:  1517
Cost:  151700
Pelanggaran:  1525
Cost:  152500
Pelanggaran:  1563
Cost:  156300
Pelanggaran:  1569
Cost:  156900
Pelanggaran:  1569
Cost:  156900
Pelanggaran:  1579
Cost:  157900
Pelanggaran:  1573
Cost:  157300
Pelanggaran:  1574
Cost:  157400
Pelanggaran:  1510
Cost:  151000
Pelanggaran:  1585
Cost:  158500
Pelanggaran:  1553
Cost:  155300
Pelanggaran:  1551
Cost:  155100
Pelanggaran:  1544
Cost:  154400
Pelanggaran:  1500
Cost:  150000
Pelanggaran:  1565
Cost:  156500
Pelanggaran:  1579
Cost:  157900
Pelanggaran:  1509
Cost:  150900
Pelanggaran:  1565
Cost:  156500
Pelanggaran:  1523
Cost:  152300
Pelanggaran:  1545
Cost:  154500
Pelanggara

# SELEKSI TURNAMEN

In [138]:
def tournament_selection(populasi, fitness):
    kandidat = random.sample(list(zip(populasi, fitness)), TOURNAMENT_SIZE)
    kandidat.sort(key=lambda x: x[1])
    return kandidat[0][0]

# MULTI-POINT CROSSOVER

In [139]:
def crossover(p1, p2, mask1):
    c1 = np.array(p1, dtype=object)
    c2 = np.array(p2, dtype=object)

    for k in range(len(c1)):
        for h in range(len(c1[k]) - 1):

            if random.random() > PROBABILITAS_CROSSOVER:
                continue

            slot_len = len(c1[k][h])
            mask_flat = np.array(mask1[k][h])

            zero_id = np.where(mask_flat == 0)[0]
            if len(zero_id) > 0 and random.random() < BIAS_CROSSOVER:
                point = np.random.choice(zero_id, size=random.randint(1, len(zero_id)), replace=False)

            else:
                point = np.random.choice(slot_len, size=random.randint(1, slot_len), replace=False)

            for id in point:
                c1[k][h][id], c2[k][h][id] = c2[k][h][id], c1[k][h][id]

    return c1.tolist(), c2.tolist()

# SCRAMBLED MUTATION

In [140]:
def mutasi(individu, mask):
    for k in range(len(individu)):
        for h in range(len(individu[k]) - 1 ):

            if random.random() > PROBABILITAS_MUTASI:
                continue
                
            hari = individu[k][h]
            hari_mask = np.array(mask[k][h])

            zero_id = np.where(hari_mask == 0)[0]

            if len(zero_id) > 1 and random.random() < BIAS_MUTASI:
                id = zero_id
            else:
                id = np.random.choice(len(hari), size=random.randint(2, len(hari)), replace=False)

            value = [hari[i] for i in id]
            random.shuffle(value)

            for i,v in zip(id, value):
                hari[i] = v
    
    return individu